In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd()
while project_root.name != 'python' and project_root.parent != project_root:
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

## Downloading the data

In [ ]:
from data import download_tickers_history
from data.constants import TRADING_DAYS_PER_YEAR

# set the date range for the historic data
start_date = datetime(year=2020, month=1, day=1)
end_date = datetime(year=2026, month=1, day=1)
tickers = ['NVDA', 'AAPL', 'PLTR', 'DKNG', 'CAT', 'INTC', 'AMZN', 'TSLA', 'GOOG', 'MSFT']
history = download_tickers_history(start_date, end_date, tickers).dropna()

start_date = datetime(year=2020, month=1, day=1)
end_date = datetime(year=2026, month=1, day=1)
tickers = ['^GSPC']
sp500_history = download_tickers_history(start_date, end_date, tickers).dropna()

history.head()


#### Backtesting logic

In [ ]:
from data import (
    RISK_UNACCEPTANCE_VALUE,
    calcualte_structural_cost_coupling_value,
    get_portfolio_exp_vol,
    log_returns,
)
from src.portfolio import find_max_sharpe

history_days = history.index.shape[0]
num_assets = history.columns.levels[0].nunique() # type: ignore

log_ret = log_returns(history)

fixed_penalty = True

lookback_window = 504
rebalance_period = 21 # 21-trading day
rebalances_per_year = int(TRADING_DAYS_PER_YEAR / rebalance_period)

backtest_res = []
portfolio_value = 10000.0 # starting capital = 10,000 $
equal_portfolio_value = 10000.0
broker_commission = 0.0005 # 0.05%

# Trader's portfolio
w_drift = None
sharpe = None
close = history.xs("Close", axis=1, level="Price")

# S&P 500
sp500_close = sp500_history.xs("Close", axis=1, level="Price")

# Equal weights
equal_w_drift = np.ones(num_assets) / num_assets

for day in range(lookback_window, history_days, rebalance_period):
    # In-Sample
    training_sample = history.iloc[day-lookback_window:day]
    start_date = training_sample.index[0]
    end_date = training_sample.index[-1]
    turnover = 1.0

    log_ret_slice = log_ret.iloc[day - lookback_window : day]
    if not fixed_penalty:
        penalty = calcualte_structural_cost_coupling_value(
            training_sample,
            log_ret_slice,
            turnover,
            portfolio_value,
            broker_commission,
            rebalances_per_year,
            w_drift
        )
    else:
        expected_annyal_vol = get_portfolio_exp_vol(log_ret_slice, w_drift)
        penalty = (RISK_UNACCEPTANCE_VALUE * 0.2) / expected_annyal_vol
    
    (sharpe, sh_weights) = find_max_sharpe(
        training_sample,
        'T_BILLS',
        'GARCH',
        'HISTORICAL',
        'BACKTEST',
        prediction_period=rebalance_period,
        init_weights=w_drift,
        l1_coeff=penalty)

    # Out-of-Sample evaluation
    out_of_sample_start = day # out-of-sample starting point
    out_of_sample_end = min(day + rebalance_period, history_days) # out-of-sample ending point

    # Ensure weights are a flat 1D array
    sh_weights = np.squeeze(sh_weights).astype(float) / 100

    # Calculate trader's portfolio deduction value
    turnover = np.abs(sh_weights - w_drift).sum() if w_drift is not None else turnover
    t_cost = turnover * broker_commission
    portfolio_value *= (1 - t_cost)    
    w_drift = sh_weights

    # Calculate equal portfolio deduction value
    eq_weights = np.ones(num_assets) / num_assets
    eq_turnover = np.abs(eq_weights - equal_w_drift).sum()
    t_eq_cost = eq_turnover * broker_commission
    equal_portfolio_value *= (1 - t_eq_cost)    
    equal_w_drift = eq_weights

    for p in range(out_of_sample_start, out_of_sample_end):
        daily_asset_returns = np.array((close.iloc[p] - close.iloc[p - 1]) / close.iloc[p - 1])
        portfolio_returns = np.dot(w_drift, daily_asset_returns)
        effective_port_ret = portfolio_returns - t_cost if p == out_of_sample_start else portfolio_returns

        portfolio_value *= (1 + portfolio_returns)
        w_drift = w_drift * ((1 + daily_asset_returns) / (1 + portfolio_returns))
        w_drift = w_drift / np.sum(w_drift)

        # Equal weights
        equal_daily_asset_returns = (close.iloc[p].values - close.iloc[p - 1].values) / close.iloc[p - 1].values # type: ignore
        equal_portfolio_returns = np.dot(equal_w_drift, equal_daily_asset_returns)
        equal_port_ret = equal_portfolio_returns - t_eq_cost if p == out_of_sample_start else equal_portfolio_returns

        equal_portfolio_value *= (1 + equal_portfolio_returns)
        equal_w_drift = equal_w_drift * ((1 + equal_daily_asset_returns) / (1 + equal_portfolio_returns))
        equal_w_drift = equal_w_drift / np.sum(equal_w_drift)

        # Period results
        backtest_res.append({
            "date": close.index[p],
            "sharpe": sharpe.max_sharpe,
            "weights": sh_weights.copy(),
            "daily_return": effective_port_ret,
            "equal_portfolio_daily_return": equal_port_ret,
            "portfolio_value": portfolio_value,
            "equal_portfolio_value": equal_portfolio_value
        })

backtest_res_df = pd.DataFrame(backtest_res).set_index("date")

sp500_oos_prices = sp500_close.loc[backtest_res_df.index]
sp500_equity = 10000 * (sp500_oos_prices / sp500_oos_prices.iloc[0])

backtest_res_df.head()


## Visualize backtesting data

#### Cumulative Equity Curve
Shows whether a complex algorithmic platform overtakes the simple passive market and naive (equal-weighted) diversification at a distance.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Trader's portfolio
date = pd.to_datetime(backtest_res_df.index)
equity = backtest_res_df["portfolio_value"]

ax.plot(
    date,
    equity,
    color="darkgreen",
    linewidth=2,
    label="Portfolio Cumulative Equity",
)

# Equals weights
eq_equity = backtest_res_df["equal_portfolio_value"]

ax.plot(
    date,
    eq_equity,
    color="darkorange",
    linewidth=1,
    label="Equal Weights Portfolio Cumulative Equity",
)

# S&P 500
ax.plot(
    date,
    sp500_equity,
    color="darkblue",
    linewidth=1,
    label="S&P 500 Cumulative Equity",
)

# Title and labels
plt.title('Cumulative Equity Curve', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Cumulative Returns ($)', fontsize=12)

# Legend box showing what each color curve represents
ax.legend(
    title="Curve Legend",
    title_fontsize=11,
    loc="upper left",
    frameon=True,
    facecolor="#ffffff",
    edgecolor="black",
    framealpha=0.95,
    fontsize=10,
)

plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

plt.show()


#### Underwater Drawdown Chart
Shows strategy stress test. The optimized portfolio should have a significantly lower depth of flow and faster recovery (Recovery Time) than the _S&P 500_ during crises.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Trader's portfolio
date = pd.to_datetime(backtest_res_df.index)
equity = backtest_res_df["portfolio_value"]
drawdown = (equity - equity.cummax()) / equity.cummax()

ax.plot(
    date,
    drawdown,
    color="darkgreen",
    linewidth=2,
    label="Portfolio Underwater Drawdown",
)

ax.axhline(0, color="gray", linestyle="-", linewidth=1, alpha=1)

# Equal weights portfolio
eq_equity = backtest_res_df["equal_portfolio_value"]
eq_drawdown = (eq_equity - eq_equity.cummax()) / eq_equity.cummax()

ax.plot(
    date,
    eq_drawdown,
    color="darkorange",
    linewidth=1,
    label="Equal Weights Portfolio Underwater Drawdown",
)

# S&P 500
sp500_drawdown = (sp500_equity - sp500_equity.cummax()) / sp500_equity.cummax()

ax.plot(
    date,
    sp500_drawdown,
    color="darkblue",
    linewidth=1,
    label="S&P 500 Underwater Drawdown",
)

# Title and labels
plt.title('Underwater Drawdown Curve', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Cumulative Returns ($)', fontsize=12)

# Legend box showing what each color curve represents
ax.legend(
    title="Curve Legend",
    title_fontsize=11,
    loc="lower left",
    frameon=True,
    facecolor="#ffffff",
    edgecolor="black",
    framealpha=0.95,
    fontsize=10,
)

plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

plt.show()


#### Weights History
Shows model stability and adequacy:
* If weights change smoothly from month to month, the model is stable.
* If the weights are chaotically raised from $0% to $100% each step - too much noise in the optimizer, transaction fees will eat up all the profits.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Trader's portfolio
weights_df = pd.DataFrame(
    (backtest_res_df["weights"] * 100).tolist(),
    index=backtest_res_df.index,
    columns=close.columns
)
date = weights_df.index
ax.stackplot(
    date,
    weights_df.T,
    linewidth=2,
    labels=weights_df.columns,
)

handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles[::-1],
    labels[::-1],
    title="Assets",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=True
)

plt.title('Weights History Stacked Area', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Weights (%)', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

plt.show()


#### Rolling Sharpe Ratio, 6M / 1Y
Shows the stability of alpha generation over time. Good strategy keeps Rolling Sharpe consistently above zero, without falling into deep downsides during adjustments.

In [ ]:
from src.portfolio import get_risk_free_rate

fig, ax = plt.subplots(figsize=(12, 6))

window = 126 # 126 days = 6 trading months
risk_free_rate = get_risk_free_rate('T_BILLS', start_date, end_date)

# Trader's portfolio
date = pd.to_datetime(backtest_res_df.index)
portfolio_ret = backtest_res_df["daily_return"].rolling(window)
portfolio_rolling_sharpe = ((portfolio_ret.mean() * TRADING_DAYS_PER_YEAR - risk_free_rate) / (portfolio_ret.std() * np.sqrt(TRADING_DAYS_PER_YEAR)))

ax.plot(
    date,
    portfolio_rolling_sharpe,
    color="darkgreen",
    linewidth=2,
    label="Portfolio Rolling Sharpe",
)

ax.axhline(0, color="gray", linestyle="--", linewidth=1, alpha=0.7)
ax.axhline(1.0, color="darkblue", linestyle=":", linewidth=1, alpha=0.5, label="Sharpe = 1.0 (Good)")

# Equal weights portfolio
eq_portfolio_ret = backtest_res_df["equal_portfolio_daily_return"].rolling(window)
eq_rolling_sharpe = ((eq_portfolio_ret.mean() * TRADING_DAYS_PER_YEAR - risk_free_rate) / (eq_portfolio_ret.std() * np.sqrt(TRADING_DAYS_PER_YEAR)))

ax.plot(
    date,
    eq_rolling_sharpe,
    color="darkorange",
    linewidth=1,
    label="Equal Weights Portfolio Rolling Sharpe",
)

# S&P 500
out_of_sample_sp500_df = sp500_history[sp500_history.index.isin(date)]
sp500_close = out_of_sample_sp500_df.xs("Close", axis=1, level="Price")
sp500_ret = (sp500_close / sp500_close.shift(1) - 1).rolling(window)
sp500_rolling_sharpe = ((sp500_ret.mean() * TRADING_DAYS_PER_YEAR - risk_free_rate) / (sp500_ret.std() * np.sqrt(TRADING_DAYS_PER_YEAR)))

ax.plot(
    date,
    sp500_rolling_sharpe,
    color="darkblue",
    linewidth=1,
    label="S&P 500 Underwater Rolling Sharpe",
)

# Title and labels
plt.title('Rolling Sharpe Ratio', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Cumulative Returns ($)', fontsize=12)

# Legend box showing what each color curve represents
ax.legend(
    title="Curve Legend",
    title_fontsize=11,
    loc="lower left",
    frameon=True,
    facecolor="#ffffff",
    edgecolor="black",
    framealpha=0.95,
    fontsize=10,
)

plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

plt.show()
